# LeetCode #1235: Maximum Profit in Job Scheduling

https://leetcode.com/problems/maximum-profit-in-job-scheduling/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^n)$ | $O(n)$ |
| **Optimal: DP + Binary Search ★** | $O(n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Recursively include or exclude each job, tracking the latest end time to avoid overlaps. Without memoization, this visits every subset — exponential and impractical beyond ~20 jobs.

### Optimal: DP + Binary Search ★
Sort jobs by end time. `dp[i]` = maximum profit achievable using any subset of the first `i` jobs. For each job `i`, binary-search for the latest job `j` whose end time $\leq$ `startTime[i]`. Then `dp[i] = max(dp[i-1], profit[i] + dp[j])`. The sorted order enables the binary search.

**Why this is better than Brute Force:** By reusing cumulative max-profit values we answer each job's optimal sub-problem in $O(\log n)$, reducing overall runtime from exponential to $O(n \log n)$.

**Constraints:**
* $1 \leq n \leq 5 \times 10^4$
* $1 \leq$ `startTime[i]` $<$ `endTime[i]` $\leq 10^9$
* $1 \leq$ `profit[i]` $\leq 10^4$

## Solutions

### C#

In [ ]:
using System;
using System.Linq;

public class Solution {
    public int JobScheduling(int[] startTime, int[] endTime, int[] profit) {
        int n = startTime.Length;
        // Sort jobs by end time so earlier-finishing jobs come first
        var jobs = Enumerable.Range(0, n)
            .OrderBy(i => endTime[i])
            .Select(i => (s: startTime[i], e: endTime[i], p: profit[i]))
            .ToArray();

        // dp[i] = max profit using any subset of the first i jobs
        int[] dp = new int[n + 1];
        for (int i = 0; i < n; i++) {
            // Binary-search for the latest job whose end <= current job's start
            int lo = 0, hi = i;
            while (lo < hi) {
                int mid = (lo + hi + 1) / 2;
                if (jobs[mid - 1].e <= jobs[i].s) lo = mid;
                else hi = mid - 1;
            }
            // Either skip this job or take it on top of the best compatible prefix
            dp[i + 1] = Math.Max(dp[i], jobs[i].p + dp[lo]);
        }
        return dp[n];
    }
}

### Python

In [ ]:
import bisect
from typing import List

class Solution:
    def job_scheduling(self, start_time: List[int], end_time: List[int], profit: List[int]) -> int:
        # Sort jobs by end time so earlier-finishing jobs come first
        jobs = sorted(zip(end_time, start_time, profit))
        end_times = [j[0] for j in jobs]

        # dp[i] = max profit using any subset of the first i jobs
        dp = [0] * (len(jobs) + 1)
        for i, (e, s, p) in enumerate(jobs):
            # Binary-search for the latest job whose end <= current job's start
            j = bisect.bisect_right(end_times, s, 0, i)
            # Either skip this job or take it on top of the best compatible prefix
            dp[i + 1] = max(dp[i], p + dp[j])
        return dp[-1]

### Go

In [ ]:
import "sort"

func jobScheduling(startTime []int, endTime []int, profit []int) int {
    n := len(startTime)
    type Job struct{ s, e, p int }
    jobs := make([]Job, n)
    for i := range jobs {
        jobs[i] = Job{startTime[i], endTime[i], profit[i]}
    }
    // Sort jobs by end time so earlier-finishing jobs come first
    sort.Slice(jobs, func(i, j int) bool { return jobs[i].e < jobs[j].e })

    ends := make([]int, n)
    for i, j := range jobs { ends[i] = j.e }

    // dp[i] = max profit using any subset of the first i jobs
    dp := make([]int, n+1)
    for i, job := range jobs {
        // Binary-search for the latest job whose end <= current job's start
        idx := sort.SearchInts(ends[:i], job.s+1) // first end > start
        // Either skip this job or take it on top of the best compatible prefix
        take := job.p + dp[idx]
        if dp[i] > take { dp[i+1] = dp[i] } else { dp[i+1] = take }
    }
    return dp[n]
}

### Rust

In [ ]:
impl Solution {
    pub fn job_scheduling(start_time: Vec<i32>, end_time: Vec<i32>, profit: Vec<i32>) -> i32 {
        let n = start_time.len();
        let mut jobs: Vec<(i32, i32, i32)> = (0..n)
            .map(|i| (end_time[i], start_time[i], profit[i]))
            .collect();
        // Sort jobs by end time so earlier-finishing jobs come first
        jobs.sort_unstable();

        let ends: Vec<i32> = jobs.iter().map(|j| j.0).collect();
        // dp[i] = max profit using any subset of the first i jobs
        let mut dp = vec![0i32; n + 1];
        for (i, &(_, s, p)) in jobs.iter().enumerate() {
            // Binary-search for the latest job whose end <= current job's start
            let j = ends[..i].partition_point(|&e| e <= s);
            // Either skip this job or take it on top of the best compatible prefix
            dp[i + 1] = dp[i].max(p + dp[j]);
        }
        dp[n]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `startTime=[1,2,3], endTime=[3,4,5], profit=[20,20,20]`
Jobs don't overlap pairwise but taking all three is impossible (job 1 ends at 3, job 2 starts at 2). Optimal: take job 1 + job 3 for profit **40**, or any two non-overlapping pair.

### 2. Slightly Complex
**Input:** `startTime=[1,2,3,3], endTime=[3,4,5,6], profit=[50,10,40,70]`
Job 4 (3→6, profit 70): best compatible predecessor is job 1 (1→3, profit 50). dp[4]=120. Answer: **120**.

### 3. Edge Case: Time Factor
**Input:** 50 000 jobs each with unique start/end times sorted in increasing order.
Every binary search executes over up to 50 000 end times, totalling $O(n \log n)$ comparisons — the maximum time-complexity scenario.

### 4. Edge Case: Space Factor
**Input:** 50 000 non-overlapping jobs.
The `dp` array stores 50 001 entries; all preceding values must be retained because any earlier job might be the optimal predecessor. $O(n)$ space is unavoidable here.

### 5. Almost-Impossible but Plausible
**Input:** `startTime=[1], endTime=[1000000000], profit=[10000]`
Only one job, spanning the entire valid time range. `dp[1] = 10000`. Binary search finds no predecessor (index 0), so profit is taken outright.